# Semantic pipeline

In [8]:
import re
import numpy as np
import pandas as pd
import stanza
from nltk.corpus import wordnet
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
# initialise sentence embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
# initialise stanza nlp pipeline
stan_pipe = stanza.Pipeline(
    processors = "tokenize, pos, lemma",
    use_gpu = False,
    verbose = False
)

Now we need a function to convert between Stanza tags (NOUN, VERB etc.) and Wordnet tags (wordnet.NOUN, wordnet.VERB etc.). Also sticking with just NOUN,  VERB, ADJ (adjective) and ADV (adverb) here as these carry the bulk of the semantic content and we want to avoid feature creep, especially when other POS categories (DET etc.) have been covered in the syntactic pipeline and we want to keep our three pipelines distinct.

In [11]:
def stanza_to_wordnet(upos):

    # check for stanza noun tag
    if upos == "NOUN":
        # if noun, return wordnet noun
        return wordnet.NOUN
    
    # repeat for other chosen tags
    elif upos == "VERB":
        
        return wordnet.VERB

    elif upos == "ADV":
        
        return wordnet.ADV
    
    elif upos == "ADJ":
        
        return wordnet.ADJ

    # if not predefined tag, return none and exclude token
    else:

        return None

Now need a cleaning function, copied form syntactic.

In [12]:
def semantic_pipe_clean(text):

    # if text is missing return a blank ""
    if pd.isna(text):
        return ""

    # convert text to string if not already
    text = str(text)

    # remove any html markup still existing
    text = re.sub(r"<[^>]+>", " ", text)

    # change any multiple spaces, tabs, newlines with " "
    text = re.sub(r"\s+", " ", text)

    # strip outside whitespace
    text = text.strip()

    # return cleaned text
    return text

Now need main semanntic feature extraction pipeline.

In [13]:
def extract_semantic_features(text):

    # run cleaning function
    text = semantic_pipe_clean(text)
    # return empty dict if no text
    if not text:
        return {}

    # run text through stanza pipeline to parse
    text = stan_pipe(text)

    # set counter to track relevent tokens (noun etc)
    rel_tokens = 0
    # track relevent tokens with >= 1 worndet synonym
    syn_tokens = 0
    # track relevent tokens with >= 1 worndet hypernym
    hyper_tokens = 0
    # track relevent tokens with >= 1 worndet hyponym
    hypo_tokens = 0

       
    ##################################
    # Begin looping through words
    ##################################
    
    # loop through sentences in text
    for sentence in text.sentences:

        # loop through words in sentence
        for word in sentence.words:

            # get POS tag from stanza
            upos = word.upos
            # and convert to wordnet tag
            wn_tag = stanza_to_wordnet(upos)
            # if not one of chosen pos categories, skip
            if wn_tag is None:
        
                continue

            # get lemmatised word
            lemma = word.lemma
            # and convert to lowercase
            lemma = lemma.lower()
            # token must be relevant as we havent skipped so increment relevant tokens
            rel_tokens += 1
            # find synsets for this particular word lemma and POS combo
            synsets = wordnet.synsets(lemma, pos = wn_tag)


            ##################################
            # Synonym density
            ##################################

            # initialise flag for if this word has at least 1 synonym
            has_synonym = False
    
            # and now loop through associated synsets of the word
            for synset in synsets:
            
                # and get lemmas in that synset
                synset_lemmas = synset.lemma_names()
                
                # now check if there are synonymns other than the original word itsefl
                for synonym in synset_lemmas:

                    # convert to lower case to compare
                    synonym = synonym.lower()
                    # if the synonym isnt the same as the original word...
                    if synonym != lemma:
                        
                        # then the original word does have a synonym
                        has_synonym = True
                        # and break, no need to look for more
                        break

                # stop looking once a synonym is found
                if has_synonym:

                    break

            # if this word has at least 1 synonym...
            if has_synonym:
                
                # increment synonym token count
                syn_tokens += 1


            ##################################
            # Hypernym density 
            ##################################

            # initialise flag for if this word has at least 1 hypernym
            has_hypernym = False

            # and now loop through associated synsets of the word
            for synset in synsets:

                # get hpyernyms of synset
                hypernyms = synset.hypernyms()
                # if there is at least one...
                if hypernyms:
                    # set flag to true to show this word has at least 1 hypernym
                    has_hypernym = True
                    # then break out
                    break
            # and if we set that flag for any one word having a hypernym
            if has_hypernym:
                # incrment words with hypernyms count
                hyper_tokens += 1


            ##################################
            # Hyponym density 
            ##################################

            # initialise flag for if this word has at least 1 hyponym
            has_hyponym = False

            # and now loop through associated synsets of the word
            for synset in synsets:

                # get hyponyms of synset
                hyponyms = synset.hyponyms()
                # if there is at least one...
                if hyponyms:
                    # set flag to true to show this word has at least 1 hyponym
                    has_hyponym = True
                    # then break out
                    break
            # and if we set that flag for any one word having a hyponym
            if has_hyponym:
                # incrment words with hyponyms count
                hypo_tokens += 1


    ##################################
    # Calculate wordnet features 
    ##################################
        
    # can now calculate synonym density as relevent words
    # that have >= 1 synonym
    if rel_tokens > 0:
        # synonym density = ratio of relevent words with at least 1 synonym
        synonym_density = syn_tokens / rel_tokens

    # if no relevent words
    else:
        # return NaN
        synonym_density = np.nan

    # can now calculate hypernym density as relevent words
    # that have >= 1 hypernym
    if rel_tokens > 0:
        # hypernym density = ratio of relevent words with at least 1 hypernym
        hypernym_density = hyper_tokens / rel_tokens

    # if no relevent words
    else:
        # return NaN
        hypernym_density = np.nan

    # can now calculate hyponym density as relevent words
    # that have >= 1 hyponym
    if rel_tokens > 0:
        # hyponym density = ratio of relevent words with at least 1 hyponym
        hyponym_density = hypo_tokens / rel_tokens

    # if no relevent words
    else:
        # return NaN
        hyponym_density = np.nan


    ##################################
    # Embeddings
    ##################################

    # create embeddings for each sentence and for whole text / document
    # get list of text from each sentence
    sentences = [sentence.text for sentence in text.sentences]
    # create an embedding for each sentence
    sent_embeddings = embedding_model.encode(sentences)
    # convert sentence embeddings to array
    sent_embeddings = np.array(sent_embeddings)
    # average sentences to get document embedding
    doc_embedding = np.mean(sent_embeddings, axis = 0)
    # reshape to use with cosine_similarity
    doc_embedding = doc_embedding.reshape(1, -1)


    ##################################
    # Adjacent sentence similarity
    ##################################

    # calculate adjacenet sentence similarities for use in mean, std
    # initialise mepty list to store similarities between adjacent sentences
    adj_similarity = []
    # loop through pairs of adjacent sentences
    for i in range(len(sent_embeddings) - 1):
        # get current sentence embedding
        cur_embedding = sent_embeddings[i]
        # get embedding of next sentence
        next_embedding = sent_embeddings[i + 1]
        # get cosine similarity between these
        similarity = cosine_similarity(
            cur_embedding.reshape(1, -1),
            next_embedding.reshape(1, -1)
        )[0][0]
        # save similarity in list
        adj_similarity.append(similarity)
        
        
    ##################################
    # Mean adjacent sentence similarity
    ##################################

    # ensure we have adjacenet similarities
    if adj_similarity:
        # calculate mean adjacent similairty beween these
        mean_adj_sent_similarity = np.mean(adj_similarity)
    
    # in instance of only one sentence
    else:
        # return nan
        mean_adj_sent_similarity = np.nan


    ##################################
    # STD adjacent sentence similarity
    ##################################

    # ensure we have adjacenet similarities
    if adj_similarity:
        # calculate STD adjacent similairty beween these
        std_adj_sent_similarity = np.std(adj_similarity)
    
    # in instance of only one sentence
    else:
        # return nan
        std_adj_sent_similarity = np.nan
        

    ##################################
    # Mean sentence-document similarity
    ##################################

    # set empty list to store similarities between each 
    # sentence and the text itself
    sent_doc_similarities = []
    # loop through each sentence embedding
    for sent_embedding in sent_embeddings:
        # and for each get similairty between that and the doc
        similarity = cosine_similarity(
            sent_embedding.reshape(1, -1),
            doc_embedding
        )[0][0]
        # append to list
        sent_doc_similarities.append(similarity)
    # finally get mean of these
    mean_sent_doc_similarity = np.mean(sent_doc_similarities)


    ##################################
    # Mean embedding variance
    ##################################
    
    # get the variance of each set of embeddings...
    embedding_variance = np.var(sent_embeddings, axis = 0)
    # and get the mean of these
    mean_embedding_variance = np.mean(embedding_variance)



    # return features
    return {
        "synonym_density" :          synonym_density, 
        "hypernym_density" :         hypernym_density, 
        "hyponym_density" :          hyponym_density,

        "mean_adj_sent_similarity" : mean_adj_sent_similarity,
        "std_adj_sent_similarity" :  std_adj_sent_similarity,
        "mean_sent_doc_similarity" : mean_sent_doc_similarity,

        "mean_embedding_variance" :  mean_embedding_variance
    }

Same test piece of text used multiple times previously.

In [8]:
test_text = """

Conservative leader Kemi Badenoch has pressed Andy Burnham over UK borrowing and spending at his first Prime Minister's Questions.
Badenoch called on the PM to say where spending would be cut and how he would deal with the UK's increasing debt, after the cost of 
borrowing for the UK rose, reaching a new 18-year high on Wednesday. The PM blamed the previous Conservative government saying that the 
"turbulence on global markets are because of that exposure that they left behind". Burnham became prime minister in July, having taken 
over from Sir Keir Starmer just before Parliament's summer recess, and it was the first time he had faced the Tory leader at the Commons session.
The prime minister used the summer recess to tour the UK, and had announced a series of eye-catching policies, mainly aimed at tackling the 
cost of living. MPs returned to Parliament on Tuesday and Burnham spent more than three hours in the Commons, making a statement and 
answering questions from MPs. At PMQs, Badenoch said the prime minister had set out plenty of spending plans but no detail of how he would be 
paying the bill. "The markets are clearly worried that we now have a spendthrift prime minister who wants to say yes to everyone but cannot 
tell us where the money is coming from," she said. "He needs to pay for all his new promises and he has a choice - higher taxes, more borrowing, 
or spending cuts. "Yesterday he said change begins with honesty, so will he be honest and tell us: is he preparing to raise taxes again, yes or no?"
Burnham had earlier said he had already cut taxes, including VAT on energy bills and a cut in business rates for hospitality. "I've already 
indicated my first moves were to cut tax, but I'm not going to do what any prime minister has done and write the budget here," he said, adding 
he and Chancellor John Healey had agreed on an early date for the Budget in October to reduce speculation. At PMQs, Badenoch also raised comments 
from economist and cross-bench peer Lord Jim O'Neill, who had been tipped to become Burnham's chief economic adviser, but who ruled out joining 
Burnham's government. Speaking to the BBC, Lord O'Neill had praised the PM for "a great first five weeks" which he said had helped lift consumer 
and business confidence; adding the current spike in market interest rates for government debt would force Labour to start "dealing with the triple 
lock" on the state pension, and "excessive" welfare spending. Badenoch pointed to a warning from Lord O'Neill that the prime minister's tone 
during his Tuesday Commons debut would have unsettled investors, although the peer had also said that spike was mostly mirroring US market 
upheaval concerning the Iran conflict.

"""

In [9]:
test = extract_semantic_features(test_text)

In [10]:
test

{'synonym_density': 0.8918918918918919,
 'hypernym_density': 0.7657657657657657,
 'hyponym_density': 0.7342342342342343,
 'mean_adj_sent_similarity': 0.35448235,
 'std_adj_sent_similarity': 0.1379442,
 'mean_sent_doc_similarity': 0.59064674,
 'mean_embedding_variance': 0.0016956677}

__________

# MAIN EXTRACTION

Now that pipeline works we want to run train text through it to convert each text to a numerical represntation, with the whole train set becoming feature matrix x_train_surface.

In [7]:
# load training data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\train_dataset_master.csv"
)

# run semantic feature extraction on first 20 texts to ensure it works as expected
X_train_semantic = pd.DataFrame(
    train_df["text"].iloc[:20].apply(extract_semantic_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(X_train_semantic.shape)
print(X_train_semantic.isna().sum().sum())
print(X_train_semantic.dtypes.value_counts())
X_train_semantic.head(10)

(20, 7)
0
float32    4
float64    3
Name: count, dtype: int64


,synonym_density,hypernym_density,hyponym_density,mean_adj_sent_similarity,std_adj_sent_similarity,mean_sent_doc_similarity,mean_embedding_variance
0,0.897778,0.733333,0.702222,0.350856,0.137179,0.531834,0.001868
1,0.890000,0.620000,0.580000,0.278304,0.120284,0.555397,0.001801
2,0.890476,0.771429,0.738095,0.230359,0.119014,0.484459,0.001993
3,0.894977,0.739726,0.684932,0.231584,0.118779,0.511674,0.001922
4,0.960000,0.635000,0.615000,0.250077,0.198352,0.472533,0.002023
5,0.892086,0.805755,0.776978,0.355919,0.150520,0.553009,0.001808
6,0.906977,0.641860,0.609302,0.273552,0.103633,0.527622,0.001879
7,0.896739,0.586957,0.565217,0.319517,0.192496,0.587991,0.001704
8,0.936441,0.665254,0.644068,0.319188,0.193460,0.576360,0.001739
9,0.941463,0.790244,0.746341,0.240414,0.121160,0.486420,0.001988


Looks good.

In [11]:
# load training data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\train_dataset_master.csv"
)

# run semantic feature extraction on entire set
X_train_semantic = pd.DataFrame(
    train_df["text"].apply(extract_semantic_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(X_train_semantic.shape)
print(X_train_semantic.isna().sum().sum())
print(X_train_semantic.dtypes.value_counts())
X_train_semantic.head(10)

# crashed, save auto, starting again at 06:20
X_train_semantic.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_semantic.csv",
    index = False
)

(3200, 7)
3
float32    4
float64    3
Name: count, dtype: int64


In [12]:
print(X_train_semantic.isna().sum())

synonym_density             1
hypernym_density            1
hyponym_density             1
mean_adj_sent_similarity    0
std_adj_sent_similarity     0
mean_sent_doc_similarity    0
mean_embedding_variance     0
dtype: int64


Given this is one of each it possibly could be one specific text resulting in all three, lets check.

In [13]:
print(X_train_semantic[
      X_train_semantic[
      ["synonym_density", "hypernym_density", "hyponym_density"]
      ].isna().any(axis = 1)
])

     synonym_density  hypernym_density  hyponym_density  \
969              NaN               NaN              NaN   

     mean_adj_sent_similarity  std_adj_sent_similarity  \
969                  0.485909                  0.10025   

     mean_sent_doc_similarity  mean_embedding_variance  
969                  0.705666                 0.001307  


In [14]:
print(train_df.loc[969, "text"])

Cafodd Paul Davies, yr Aelod Cynulliad Ceidwadol dros Preseli, Sir Benfro, y cyfle i lunio deddf newydd ar awtistiaeth ar ôl iddo ennill pleidlais yn y Senedd fis Mawrth. Pwrpas y ddeddfwriaeth yw cynnig mwy o gefnogaeth i bobl sy'n byw gyda'r cyflwr. Mae'r mesur yn anelu at gyflwyno strategaeth ar gyfer diwallu anghenion plant ac oedolion yng Nghymru, a sicrhau bod cynghorau a byrddau iechyd yn deall ac yn cymryd y camau angenrheidiol er mwyn i blant ac oedolion dderbyn y cymorth sydd ei angen arnynt. Bydd pleidlais ar y Ddeddf Awtistiaeth yng Nghymru yn mynd ger bron aelodau'r cynulliad ddydd Mercher. Mabwysiadodd Cymru strategaeth awtistiaeth yn 2008 ac yn ddiweddar cafodd y strategaeth ei diweddaru i gynnwys targed amser aros penodol. Yr argymhelliad yw na fydd yn rhaid i blant ag awtistiaeth aros mwy na 26 wythnos cyn cael yr apwyntiad arbenigol cyntaf. Mae'r cynllun presennol ond yn para tan 2020. "Be mae fy mil i'n ei wneud," medd Paul Davies, "yw sicrhau bod y gwasanaethau yma 

Looks like a Welsh BBC article slipped through the net. Need to remove this from all sets.

In [16]:
print(train_df.loc[969, ["document_id", "source", "topic", "label", "author"]])

document_id    HU_BBC_0141
source                 BBC
topic              General
label                    0
author               human
Name: 969, dtype: object


I now know I can remove HU_BBC_0141 from the training data master set and all three x train.


In [18]:
X_train_semantic = X_train_semantic.drop(index = 969).reset_index(drop = True)

In [19]:
# check output
print(X_train_semantic.shape)
print(X_train_semantic.isna().sum().sum())
print(X_train_semantic.dtypes.value_counts())

(3199, 7)
0
float32    4
float64    3
Name: count, dtype: int64


Happy now, can finally save.

In [21]:
X_train_semantic.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_semantic.csv",
    index = False
)

__________

Now can do same for surface, syntactic and training main. All features extractions were run the same way - for text in training... and so it will be document 969  in all.

### Drop row for surface

In [23]:
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_surface.csv"
)

In [24]:
# check output
print(train_df.shape)
print(train_df.isna().sum().sum())
print(train_df.dtypes.value_counts())

(3200, 96)
0
float64    93
int64       3
Name: count, dtype: int64


In [27]:
train_df = train_df.drop(index = 969).reset_index(drop = True)

In [28]:
# check output
print(train_df.shape)
print(train_df.isna().sum().sum())
print(train_df.dtypes.value_counts())

(3199, 96)
0
float64    93
int64       3
Name: count, dtype: int64


In [29]:
train_df.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_surface.csv",
    index = False
)

___

### Drop row for syntactic

In [30]:
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_syntactic.csv"
)

In [31]:
# check output
print(train_df.shape)
print(train_df.isna().sum().sum())
print(train_df.dtypes.value_counts())

(3200, 96)
0
float64    96
Name: count, dtype: int64


In [32]:
train_df = train_df.drop(index = 969).reset_index(drop = True)

In [33]:
# check output
print(train_df.shape)
print(train_df.isna().sum().sum())
print(train_df.dtypes.value_counts())

(3199, 96)
0
float64    96
Name: count, dtype: int64


In [34]:
train_df.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_syntactic.csv",
    index = False
)

___

### Drop row for master train dataset

In [35]:
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\train_dataset_master.csv"
)

In [39]:
# check output
print(train_df.shape)
print(train_df.isna().sum().sum())
print(train_df.dtypes.value_counts())

(3200, 7)
0
object     5
float64    1
int64      1
Name: count, dtype: int64


In [40]:
train_df = train_df.drop(index = 969).reset_index(drop = True)

In [41]:
# check output
print(train_df.shape)
print(train_df.isna().sum().sum())
print(train_df.dtypes.value_counts())

(3199, 7)
0
object     5
float64    1
int64      1
Name: count, dtype: int64


In [43]:
train_df.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\train_dataset_master.csv",
    index = False
)

____

Welsh text now dropped from all sets.

___

# Returning for Objective 5

Had to re-run a 300 test set through paraphrasing and so now have a slightly reduced train set of 2900 that must have features extracted again so that training doesn't train on features that are now in the test set. New train set is:

C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_train_2900.csv

Will now re-run on the new train set.

In [7]:
# load training data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_train_2900.csv"
)

# run semantic feature extraction on entire set
X_train_semantic_2900 = pd.DataFrame(
    train_df["text"].apply(extract_semantic_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(X_train_semantic_2900.shape)
print(X_train_semantic_2900.isna().sum().sum())
print(X_train_semantic_2900.dtypes.value_counts())
X_train_semantic_2900.head(10)

X_train_semantic_2900.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_semantic_2900.csv",
    index = False
)

(2899, 7)
0
float32    4
float64    3
Name: count, dtype: int64


All good, the 2899 is due to the Welsh text removed previously. As expected, moving on.

___

Now running the same for the test data that has been split into light heavy etc.

In [14]:
# load test data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_heavy_300.csv"
)

# run semantic feature extraction on entire set
test_heavy_300 = pd.DataFrame(
    train_df["text"].apply(extract_semantic_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(test_heavy_300.shape)
print(test_heavy_300.isna().sum().sum())
print(test_heavy_300.dtypes.value_counts())
test_heavy_300.head(10)

test_heavy_300.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_heavy_300_features_semantic.csv",
    index = False
)

(300, 7)
0
float32    4
float64    3
Name: count, dtype: int64


In [15]:
# load test data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_light_300.csv"
)

# run semantic feature extraction on entire set
test_light_300 = pd.DataFrame(
    train_df["text"].apply(extract_semantic_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(test_light_300.shape)
print(test_light_300.isna().sum().sum())
print(test_light_300.dtypes.value_counts())
test_light_300.head(10)

test_light_300.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_light_300_features_semantic.csv",
    index = False
)

(300, 7)
0
float32    4
float64    3
Name: count, dtype: int64


In [16]:
# load test data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_original_300.csv"
)

# run semantic feature extraction on entire set
test_original_300 = pd.DataFrame(
    train_df["text"].apply(extract_semantic_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(test_original_300.shape)
print(test_original_300.isna().sum().sum())
print(test_original_300.dtypes.value_counts())
test_original_300.head(10)

test_original_300.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_original_300_features_semantic.csv",
    index = False
)

(300, 7)
0
float32    4
float64    3
Name: count, dtype: int64


In [17]:
# load test data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_trans_300.csv"
)

# run semantic feature extraction on entire set
test_trans_300 = pd.DataFrame(
    train_df["text"].apply(extract_semantic_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(test_trans_300.shape)
print(test_trans_300.isna().sum().sum())
print(test_trans_300.dtypes.value_counts())
test_trans_300.head(10)

test_trans_300.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_trans_300_features_semantic.csv",
    index = False
)

(300, 7)
0
float32    4
float64    3
Name: count, dtype: int64


All done again.